# Notebook 07: Fixed Anatomical Graph Neural Network (Fixed GNN)

**Objective**: Train and evaluate the Fixed Anatomical Graph Neural Network (`FixedAnatomicalGNN`).

### Architecture Overview:
1. **Regional Temporal Encoders**: Separate 1D Conv/BatchNorm/Pooling blocks for each of the 5 body regions, transforming variable-channel sensor streams into uniform $D$-dimensional node representations ($D=64$).
2. **Fixed Graph Convolution**: Standard Kipf-Welling GCN message passing over the static skeletal adjacency matrix $\tilde{A}_{\text{fixed}}$ connecting limbs to the trunk.
3. **Global Graph Readout**: Concatenated mean and max pooling across all 5 anatomical nodes followed by an MLP classifier.

> **Compute Rule**: Full training requires CUDA GPU (Colab). Set `EXPERIMENT_MODE = 'DEBUG'` for 1-batch CPU verification.

## 0. Environment Setup & Repository Bootstrap

In [ ]:
import sys
import os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_DIR = Path("/content/Opportunity_HAR")
    if not REPO_DIR.exists():
        print("Cloning Opportunity_HAR from GitHub...")
        !git clone https://github.com/DhruvikaRajput/Opportunity_HAR.git {REPO_DIR}
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    os.chdir(REPO_DIR)
    !pip install -q -r requirements.txt
else:
    current = Path.cwd()
    REPO_DIR = current.parent if current.name == "notebooks" else current
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    os.chdir(REPO_DIR)

import src.api as api
env_info = api.setup_project()
print(f"Project Initialized on: {env_info['device_name']} (CUDA: {env_info['cuda_available']})")

## 1. Configure Experiment Mode

In [ ]:
# Choose: 'DEBUG' (1-epoch CPU/GPU check), 'FAST' (5 epochs GPU), or 'FULL' (50 epochs GPU)
EXPERIMENT_MODE = "FAST" if env_info["cuda_available"] else "DEBUG"
print(f"Selected EXPERIMENT_MODE: {EXPERIMENT_MODE}")
if not env_info["cuda_available"] and EXPERIMENT_MODE != "DEBUG":
    print("Notice: CUDA is not available. Please switch Colab to GPU (Runtime > Change runtime type > T4 GPU) or use DEBUG mode.")

## 2. Load Processed Sliding Windows & DataLoaders

In [ ]:
try:
    subset = api.load_preprocessed_subset("opportunity_locomotion_subset.npz")
    X_train, y_train = subset["X_train"], subset["y_train"]
    X_val, y_val = subset["X_val"], subset["y_val"]
    X_test, y_test = subset["X_test"], subset["y_test"]

    if EXPERIMENT_MODE == "DEBUG":
        X_train, y_train = X_train[:64], y_train[:64]
        X_val, y_val = X_val[:32], y_val[:32]
        X_test, y_test = X_test[:32], y_test[:32]

    train_loader, val_loader, test_loader = api.create_dataloaders(
        X_train=X_train, y_train=y_train,
        X_val=X_val, y_val=y_val,
        X_test=X_test, y_test=y_test,
        batch_size=64 if EXPERIMENT_MODE != "DEBUG" else 16,
    )
    print(f"DataLoaders: {len(train_loader)} train batches, {len(val_loader)} val batches.")
except FileNotFoundError:
    print("[NOTICE] Preprocessed dataset not found. Run Notebook 02 to generate opportunity_locomotion_subset.npz.")

## 3. Instantiate Fixed Anatomical GNN

In [ ]:
fixed_gnn = api.build_fixed_gnn(
    num_classes=5,
    sequence_length=30,
    node_dim=64,
    gcn_hidden_dim=64,
    dropout=0.2
)

total_params = sum(p.numel() for p in fixed_gnn.parameters() if p.requires_grad)
print(f"FixedAnatomicalGNN Initialized! Trainable parameters: {total_params:,}")

## 4. Train Model (Automatic GPU/CUDA Guard)

In [ ]:
try:
    best_fixed_gnn, history = api.train_model(
        model=fixed_gnn,
        train_loader=train_loader,
        val_loader=val_loader,
        mode=EXPERIMENT_MODE,
        lr=1e-3,
        experiment_name="fixed_anatomical_gnn"
    )
    print("Training Complete!")
except Exception as e:
    print("Training message:", e)

## 5. Evaluate on Held-Out Test Set

In [ ]:
try:
    test_metrics = api.evaluate_model(best_fixed_gnn, test_loader)
    print("\n" + "=" * 55)
    print(" FIXED ANATOMICAL GNN TEST EVALUATION")
    print("=" * 55)
    print(f"Accuracy    : {test_metrics['accuracy'] * 100:.2f}%")
    print(f"Macro F1    : {test_metrics['macro_f1'] * 100:.2f}%")
    print(f"Weighted F1 : {test_metrics['weighted_f1'] * 100:.2f}%")
    print("=" * 55)
except Exception as e:
    print("Evaluation message:", e)